## Running LYNX

In [ ]:
import os
import gc
import sys
import time

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

import IO, utils, plot, configs, dataset, trajectory
from models import vgae, model_train

### VGAE (Xenium-DESI integration)

In [ ]:
# Load paired Xenium & DESI
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
metadata_path = '../data/sample_metadata.csv'
n_desi = 100

sample_id = 'NIH_F1'
adata = IO.load_multiomics(
    sample_id, xenium_path, desi_path, 
    n_features=n_desi,
    project=True   # Project raw DESI pixel to mapped adata cells as the aux. variable
)


#### Model training: 

In [ ]:
graph_data = dataset.XeniumDataset(adata, k=30, n_subgraphs=8)
train_data, val_data = random_split(graph_data, [0.8, 0.2])
train_dl = DataLoader(graph_data, shuffle=True)
val_dl = DataLoader(graph_data)

In [ ]:
from importlib import reload
reload(IO)
reload(utils)
reload(plot)
reload(configs)
reload(dataset)
reload(vgae)
reload(model_train)


In [ ]:
torch.manual_seed(0)
device = torch.device('cuda')
lr = 1e-3

n_epochs = 500
n_hidden = 64
n_embedding = 8
n_latent = 6

train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, gamma=1.,  
    patience=20, device=device,
)

model_configs = configs.set_model_configs(
    c_in=adata.shape[1], c_aux=n_desi,
    c_hidden=n_hidden, c_latent=n_latent,
    beta=1., k_hop=2, dropout=0.5, act=nn.SiLU(),
) 

In [ ]:
torch.cuda.empty_cache()
pyro.clear_param_store()

model = vgae.VGAE(model_configs, device=train_configs.device)
model.fit(
    train_configs=train_configs,
    train_dl=train_dl,
    val_dl=val_dl,
    DEBUG=True
)

In [ ]:
def plot_factor_corr(z):
    from scipy.special import comb
    z_corr = np.corrcoef(z.T)
    z_score = np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2)

    g = sns.clustermap(z_corr, cmap='RdBu_r')
    g.figure.suptitle(
        'q(z)\n Correlation score: {}'.format(np.round(z_score, 3)), 
        fontsize=30, y=1.05
    )
    plt.show()

In [ ]:
pyro.clear_param_store()
torch.cuda.empty_cache()
device = torch.device('cpu')
k = 30
res = model.evaluate(adata, k=k, n_subgraphs=1, device=device)
adata.obsm['X_z'] = res.qz

In [ ]:
# Plot overlapped 
def plot_overlap_latent(umap1, umap2, n_samples=5000):
    n_points = umap1.shape[0]
    indices = np.random.choice(np.arange(n_points), n_samples, replace=False)
    umap_df = pd.DataFrame(
        np.vstack([umap1[indices], umap2[indices]]),
        columns=['UMAP1', 'UMAP2']
    )
    umap_df['label'] = ['Prior']*n_samples + ['Posterior']*n_samples
    sns.kdeplot(
        data=umap_df, x='UMAP1', y='UMAP2', hue='label',
        levels=5, alpha=.5, fill=True
    )
    plt.title('Prior vs. Posterior manifolds')
    plt.show()

In [ ]:
z_labels = ['z'+str(i) for i in range(n_latent)]

adata_pz = sc.AnnData(preds.pz)
adata_pz.obs[z_labels] = preds.pz
sc.pp.neighbors(adata_pz, n_neighbors=30)
sc.tl.umap(adata_pz)

adata_qz = sc.AnnData(preds.qz)
adata_qz.obs[z_labels] = preds.qz
sc.pp.neighbors(adata_qz, n_neighbors=30)
sc.tl.umap(adata_qz)

VAE w/ normal conditional prior:

In [ ]:
sc.pl.umap(adata_pz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
sc.pl.umap(adata_qz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
plot_overlap_latent(adata_pz.obsm['X_umap'], adata_qz.obsm['X_umap'])

VAE w/ GPCA conditional mean & var:

In [ ]:
sc.pl.umap(adata_pz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
sc.pl.umap(adata_qz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
plot_overlap_latent(adata_pz.obsm['X_umap'], adata_qz.obsm['X_umap'])

VAE w/ Normalizing flow:

In [ ]:
sc.pl.umap(adata_pz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
sc.pl.umap(adata_qz, color=z_labels, cmap='turbo', ncols=3, size=10)

In [ ]:
plot_overlap_latent(adata_pz.obsm['X_umap'], adata_qz.obsm['X_umap'])

In [ ]:
# VGAE w/ conditional 
display_trajectory = True
k = 30
n_zones = 5
dist_metric = 'euclidean'


if display_trajectory:
    # Factor disentanglement
    plot_factor_corr(preds.qz)
        
    # Trajectory inference
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_latent,
        dist_metric=dist_metric,
    )
    
    plot.disp_trajectory(
        adata, 
        cmap='RdBu',
        title='Spatial Gradients\n LYNX'
    )
    
    if 'milestones_colors' in adata.uns_keys():
        adata.uns.pop('milestones_colors')

    sq.pl.spatial_scatter(
        adata, color='t', 
        cmap='RdBu', size=20, img=False,
        title='Pseudotime\n'+'LYNX'
    )
    plt.show()

    utils.get_zonations(adata, n_zones=n_zones, show=True)
    sq.pl.spatial_scatter(
        adata, color='milestones', 
        cmap='RdBu_r', size=20, img=False, 
        title='Zonation\n'+'LYNX'
    )
    plt.show()

gc.collect()


In [ ]:
# VGAE without conditional 
display_trajectory = True
k = 30
n_zones = 5
n_subgraphs = 1
device = torch.device('cpu')
dist_metric = 'knn'
 
pyro.clear_param_store()
torch.cuda.empty_cache()

preds = model.evaluate(adata, k=k, n_subgraphs=n_subgraphs, device=device)
adata.obsm['X_z'] = preds.qz

if display_trajectory:
    
    # Factor disentanglement
    plot_factor_corr(preds.qz)
        
    # Trajectory inference
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_latent,
        dist_metric=dist_metric,
    )
    
    plot.disp_trajectory(
        adata, 
        cmap='RdBu',
        title='Spatial Gradients\n LYNX'
    )
    
    if 'milestones_colors' in adata.uns_keys():
        adata.uns.pop('milestones_colors')

    sq.pl.spatial_scatter(
        adata, color='t', 
        cmap='RdBu', size=20, img=False,
        title='Pseudotime\n'+'LYNX'
    )
    plt.show()

    utils.get_zonations(adata, n_zones=n_zones, show=True)
    sq.pl.spatial_scatter(
        adata, color='milestones', 
        cmap='RdBu_r', size=20, img=False, 
        title='Zonation\n'+'LYNX'
    )
    plt.show()

gc.collect()


In [ ]:
sq.pl.spatial_scatter(
    adata, color='GPX2', cmap='Blues', size=20, img=False
)

sq.pl.spatial_scatter(
    adata, color='DPT', cmap='Blues', size=20, img=False
)

In [ ]:
x_true = adata.X.A.flatten()
x_reconst = preds.px.flatten()

plt.figure(figsize=(4, 4))
plt.scatter(x_true, x_reconst, s=.5, alpha=.5)
plt.xlabel('x_true')
plt.ylabel('x_reconst')
plt.show()

In [ ]:
# Plot individual `pz`
# pz = preds.pz
# pz_labels = ['z'+str(i) for i in range(pz.shape[1])]
# for i in range(len(pz_labels)):
#     adata.obs[pz_labels[i]] = pz[:, i]

# sq.pl.spatial_scatter(
#     adata,
#     color=pz_labels,
#     img=False, size=20, ncols=4,
#     cmap='Reds'
# )
# plt.show()

# Plot individual `qz`
qz = preds.qz
qz_labels = ['z'+str(i) for i in range(qz.shape[1])]
for i in range(len(qz_labels)):
    adata.obs[qz_labels[i]] = qz[:, i]

sq.pl.spatial_scatter(
    adata,
    color=qz_labels,
    img=False, size=20, ncols=4,
    cmap='Reds'
)
plt.show()

In [ ]:
np.save('../results/lynx_{0}_attn.npy'.format(n_latent), adata.obsm['X_z'])

In [ ]:
# torch.save(model.state_dict(), '../results/multi_cat.pt'.format(n_latent)) 
# torch.save(model.state_dict(), '../results/multi_attn.pt')                                    
# del adatas, train_data, train_dl
# gc.collect()

#### All-samples

In [ ]:
from importlib import reload
reload(vgae)

In [ ]:
from scipy.special import comb

def plot_factor_corr(z):
    z_corr = np.corrcoef(z.T)
    z_score = np.abs(np.tril(z_corr, k=-1)).sum() / comb(z_corr.shape[0], 2)

    g = sns.clustermap(z_corr, cmap='RdBu_r')
    g.figure.suptitle(
        'q(z)\n Correlation score: {}'.format(np.round(z_score, 3)), 
        fontsize=30, y=1.05
    )
    plt.show()


In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
sample_ids = sorted([
    sample_id for sample_id in os.listdir(xenium_path)
    if os.path.isdir(os.path.join(xenium_path, sample_id))
])

In [ ]:
torch.manual_seed(0)
device = torch.device('cuda')
lr = 1e-3
n_epochs = 300

n_obs = 377
n_desi = 100
n_hidden = 32
n_latent = 5
n_zones = 5

k = 30
train_configs = configs.set_train_configs(
    n_epochs=n_epochs, lr=lr, gamma=1.,  
    patience=20, device=device
)

In [ ]:
# Training individual samples 

n_subgraphs = 8
n_nodes = 10
dist_metric = 'euclidean'

for sample_id in sample_ids:
    print('Training on {}...'.format(sample_id))

    # ---------------
    # Loading data
    # ---------------
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi,
        project=True
    )

    graph_data = dataset.XeniumDataset(adata, k=k, n_subgraphs=n_subgraphs)
    train_data, val_data = random_split(graph_data, [0.8, 0.2])
    train_dl = DataLoader(graph_data, shuffle=True)
    val_dl = DataLoader(graph_data)

    model_configs = configs.set_model_configs(
        c_in=n_obs, c_aux=n_desi,
        c_hidden=n_hidden, c_latent=n_latent,
        beta=1., k_hop=2, dropout=0.5, act=nn.SiLU(),
        
        w_init=utils.get_indep_components(adata.obsm['X_aux'], n_components=n_hidden)
    ) 

    # ---------------
    # Training
    # ---------------
    model = vgae.VGAE(model_configs, device=train_configs.device)
    model.fit(
        train_configs=train_configs,
        train_dl=train_dl,
        val_dl=val_dl,
        DEBUG=True
    )
    
    # ---------------------------
    # Inference & visualization
    # ---------------------------
    preds = model.evaluate(adata, k=k, n_subgraphs=1, device=torch.device('cpu'))
    adata.obsm['X_z'] = preds.qz

    del model
    gc.collect()
    torch.cuda.empty_cache()
    pyro.clear_param_store()

    
    # Factor disentanglement
    plot_factor_corr(preds.qz)

    # Trajectory inference
    try:
        trajectory.compute_trajectory(
            adata, 
            use_rep='X_z',
            n_nodes=n_nodes,
            dist_metric=dist_metric
        )
        
        plot.disp_trajectory(
            adata, 
            cmap='RdBu',
            title='Spatial Gradients\n LYNX'
        )
        
        if 'milestones_colors' in adata.uns_keys():
            adata.uns.pop('milestones_colors')
            
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime {}\n'.format(sample_id)+'LYNX'
        )
        plt.show()

        utils.get_zonations(adata, n_zones=n_zones, show=True)
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='RdBu_r', size=20, img=False, 
            title='Zonation\n'+'LYNX'
        )
        plt.show()

        time.sleep(5)
        
        # Saving results
        np.save('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id), adata.obsm['X_z'])
        del adata
        
    except ValueError:
        continue

    gc.collect()
    
    print('=============\n\n')

---